## Short Project Description

- Due Feb. 27 11:59pm
- A good time to spend thinking about your final project
- Opportunity to get feedback on the idea and to think about feasibility/ time
- Even if we didn't cover a topic you're interested in, write an idea around it and I'll guide you on its feasibility.
- Come to office hours or slack me before due date and I can explain what we will do
- If you don't have a group yet, use Slack to figure things out
- Or ask now!

Please submit your empirical project's 1-page description. In your description, please address the following points:

1. What is the research question? Clearly and succinctly articulate the precise question your research project intends to answer. Please state this as a question (with a question mark) to clarify exactly what you are trying to answer.

2. What are the context, relevance, and motivation? Provide a short discussion of why you think this question is important. What are you interested in this? Why should others be interested in this question? Understanding why something is relevant generally requires some understanding of the context of a particular literature, so please provide some limited background if that helps you better articulate your motivation.

3. What datasets are needed? Provide a list of the dataset you have identified that you think you will be using in the project. It is fine if you don’t have settled on these yet. Providing a sense of what the dataset is for is also important. If we haven't covered a topic for data extraction yet, but you would like to use it, write it down and I'll comment on its feasibility.

4. What is your methodology or identification strategy? Provide an overview of the methods you will be using to answer the research question. Are you estimating a regression model?  If so, what is your identification strategy? Are you training a machine learning model? If so, tell me what you will be predicting. Are you running an optimization model or is your study mainly descriptive in nature?

5. What is your progress status and your goal for this class project? How far are you into this project?  If so, please describe this and what you need to move forward. Also, be explicit about what you hope to achieve for this project in the course and the presentation context.

# Web Scraping with BeautifulSoup

- Web scraping is a big part of getting novel data
- It used to be easy to access webpages just with html parsers like `BeautifulSoup`
- However, with recent paywalls and javascript being used everywhere, it is no longer as easy to access web resources without increasing the machinery
- So in a sense, today's lecture is kind of less about web-scraping and more about the beauty of APIs.
    - Web scraping is still possible, but it often needs other tools, such as `selenium`, which actually creates a virtual browser that you can simulate clicks with.
    - We can totally do this in our special topics!
- 
- In any case, whether through `BeautifulSoup` or through another library, you still need to understand a little bit of html



Let's look at how to install beautiful soup and then this will be our mission:



## Very Fast HTML 

- HTML is the typesetting language of the web
- It's almost like Latex for browsers
- Nowadays, a lot of the html is created automatically through other coding languages like javascript.
- It isn't important to know the fine details, but for our purposes, HTML comes with tags that have attributes
- When we scrape, we are looking for particular tags, and to hone in on what we want, we usually narrow our search by searching for particular attributes.
- Attributes might be created from CSS, a language that is responsible for the layout and format of the HTML. 
    - So we might search for a particular tag in HTML that has CSS attributes that are from a menu, for example.

![](images/html5_cheat_sheet_tags.png)

First install beautiful soup:

`pip install bs4` 

or

`conda install bs4` (probably)

In [65]:
from bs4 import BeautifulSoup

import re  ## For parsing

import requests  ## For getting the HTML

from wordcloud import WordCloud, STOPWORDS  ## For fun

import backoff

import matplotlib.pyplot as plt

import json


### Intro to BeautifulSoup

BeautifulSoup takes the html from webpage and turns into an object that you can work with. All the tags and classes that a webpage have become attributes in a `Soup` object.

In [66]:
html_doc = """
<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>

<p class="story">Once upon a time there were three little sisters; and their names were
<a href="http://example.com/elsie" class="sister" id="link1">Elsie</a>,
<a href="http://example.com/lacie" class="sister" id="link2">Lacie</a> and
<a href="http://example.com/tillie" class="sister" id="link3">Tillie</a>;
and they lived at the bottom of a well.</p>

<p class="story">...</p>
"""

In [67]:
soup = BeautifulSoup(html_doc)
soup

<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>
<p class="story">Once upon a time there were three little sisters; and their names were
<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
<a class="sister" href="http://example.com/lacie" id="link2">Lacie</a> and
<a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>;
and they lived at the bottom of a well.</p>
<p class="story">...</p>
</body></html>

In [68]:
soup.html

<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>
<p class="story">Once upon a time there were three little sisters; and their names were
<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
<a class="sister" href="http://example.com/lacie" id="link2">Lacie</a> and
<a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>;
and they lived at the bottom of a well.</p>
<p class="story">...</p>
</body></html>

In [69]:
soup.head

<head><title>The Dormouse's story</title></head>

In [70]:
soup.title

<title>The Dormouse's story</title>

In [71]:
soup.p

<p class="title"><b>The Dormouse's story</b></p>

In [72]:
soup.a

<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>

In [73]:
soup.find_all("a")

[<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
 <a class="sister" href="http://example.com/lacie" id="link2">Lacie</a>,
 <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>]

## Now real website!

Let's look at the New york times politics section:

https://www.nytimes.com/section/politics

Seems like a nice look to work from. Let's go to the browser and check what we can see?

In [138]:
## Request the url

url = "https://www.nytimes.com/section/politics"

r = requests.get(
    url,
)
r

<Response [200]>

In [102]:
r.text

'<!DOCTYPE html>\n<html lang="en"  data-nyt-compute-assignment="fallback" xmlns:og="http://opengraphprotocol.org/schema/">\n  <head>\n    \n    \n    <meta charset="utf-8" />\n    <title data-rh="true">U.S. Politics - The New York Times</title>\n    <meta data-rh="true" property="og:description" content="Breaking news and analysis on U.S. politics, including the latest coverage of the White House, Congress, the Supreme Court and more."/><meta data-rh="true" name="description" content="Breaking news and analysis on U.S. politics, including the latest coverage of the White House, Congress, the Supreme Court and more."/><meta data-rh="true" property="twitter:description" name="description" content="Breaking news and analysis on U.S. politics, including the latest coverage of the White House, Congress, the Supreme Court and more."/><meta data-rh="true" property="og:title" content="U.S. Politics"/><meta data-rh="true" property="twitter:title" content="U.S. Politics"/><meta data-rh="true" pr

In [103]:
soup = BeautifulSoup(r.text)

In [104]:
soup

<!DOCTYPE html>
<html data-nyt-compute-assignment="fallback" lang="en" xmlns:og="http://opengraphprotocol.org/schema/">
<head>
<meta charset="utf-8"/>
<title data-rh="true">U.S. Politics - The New York Times</title>
<meta content="Breaking news and analysis on U.S. politics, including the latest coverage of the White House, Congress, the Supreme Court and more." data-rh="true" property="og:description"/><meta content="Breaking news and analysis on U.S. politics, including the latest coverage of the White House, Congress, the Supreme Court and more." data-rh="true" name="description"/><meta content="Breaking news and analysis on U.S. politics, including the latest coverage of the White House, Congress, the Supreme Court and more." data-rh="true" name="description" property="twitter:description"/><meta content="U.S. Politics" data-rh="true" property="og:title"/><meta content="U.S. Politics" data-rh="true" property="twitter:title"/><meta content="https://static01.nyt.com/newsgraphics/imag

In [105]:
url_list = []

needed_divs = soup.find_all("article")
for div in needed_divs:
    url_list.append(div.a.get("href"))

url_list


['/2026/02/18/us/politics/pirro-inquiry-video-democratic-lawmakers.html',
 '/2026/02/17/us/politics/republicans-vote-fraud-id-midterms.html',
 '/interactive/2026/02/18/us/politics/midterms-house-maps-redistricting.html',
 '/2026/02/18/us/politics/democrats-trump-congress.html',
 '/2026/02/18/us/politics/bhattacharya-kennedy-cdc-director.html',
 '/2026/02/18/technology/meta-65-million-election-ai.html',
 '/2026/02/18/us/politics/colorado-redistricting-house-map-2028.html',
 '/2026/02/18/us/politics/pirro-inquiry-video-democratic-lawmakers.html',
 '/2026/02/18/health/fda-moderna-flu-vaccine-mrna.html',
 '/2026/02/18/business/energy-environment/uber-electric-vehicle-charging-stations.html',
 '/2026/02/18/us/politics/dan-helmer-congress-redistricting-virginia.html',
 '/2026/02/18/us/politics/democrats-trump-congress.html',
 '/interactive/2026/02/18/us/politics/midterms-house-maps-redistricting.html',
 '/live/2026/02/18/us/trump-news/hud-opens-discrimination-investigation-into-a-muslim-deve

Now let's go to each website and do our searches for each candidate. But wait!

### A brief foray into decorators and the `backoff` module

Oftentimes, when you loop through many webpages, the loop might break because you're making too many requests at once. 

That's what `backoff` is for. If the website gives an error, `backoff` will catch the exception and make the request again with some pause. The more times the exception is thrown the longer `backoff` will pause before making another one. 

`backoff` works as a decorator function. What's that? It's basically a function takes a function as an argument, but returns some "wrapper" for that function that references a function in it... What does that mean?

In [79]:
def add():
    print("1+2=3")


add()


1+2=3


But now you wanted to make sure that people knew that this was a function about addition when they called it. You can write something like this:

In [80]:
def i_want_everyone_to_understand(func):

    def that_this_is_addition():
        print("just in case you didn't know, this is addition")
        func()

    return that_this_is_addition


add = i_want_everyone_to_understand(add)

add()

just in case you didn't know, this is addition
1+2=3


Instead of writing out the whole function, we can do:


In [81]:
@i_want_everyone_to_understand
def add():
    print("1+2=3")


add()

just in case you didn't know, this is addition
1+2=3


So `backoff` does this samething, but it catches exceptions of your function. So let's make out requests getter a function so we can use `backoff` with it.

In [141]:
def backoff_hdlr(details):
    print(
        "Backing off {wait:0.1f} seconds after {tries} tries "
        "calling function {target} with args {args} and kwargs "
        "{kwargs}".format(**details)
    )


@backoff.on_exception(
    backoff.fibo, requests.exceptions.RequestException, on_backoff=backoff_hdlr
)
def requester(url):
    return requests.get(
        url,
    )


In [83]:
# Test how this works

requester("")

Backing off 0.8 seconds after 1 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}
Backing off 0.3 seconds after 2 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}
Backing off 0.3 seconds after 3 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}
Backing off 0.9 seconds after 4 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}
Backing off 4.8 seconds after 5 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}
Backing off 3.5 seconds after 6 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}
Backing off 1.5 seconds after 7 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}
Backing off 4.2 seconds after 8 tries calling function <function requester at 0x11045e320> with args ('',) and kwargs {}


KeyboardInterrupt: 

### Terms of Service and the Legality of Web Scraping

- Before scraping any website, **always check its Terms of Service (ToS)** — many explicitly prohibit automated data collection
- The legal landscape is evolving, but a few landmark cases are worth knowing:
    - **hiQ Labs v. LinkedIn (2022)**: The Ninth Circuit ruled that scraping *publicly available* data does not violate the Computer Fraud and Abuse Act (CFAA). This was a major win for scrapers of public data.
    - **Van Buren v. United States (2021)**: The Supreme Court narrowed the CFAA, ruling that "exceeding authorized access" means accessing data you aren't entitled to — not simply using permitted data in an unauthorized way.
    - However, violating a website's ToS can still expose you to **breach of contract** claims, even if it doesn't trigger the CFAA
- **`robots.txt`**: Most websites have a `robots.txt` file (e.g., `https://nytimes.com/robots.txt`) that tells automated crawlers which parts of the site they're allowed to access. It's not legally binding, but ignoring it is considered bad practice and can be used as evidence of bad faith.
- **Copyright still applies**: Even if you can technically scrape it, the *content* you collect may be copyrighted. Scraping data for personal research is usually fine, but redistributing scraped content (articles, images, etc.) can get you into trouble.
- **Bottom line**: Legal ≠ ethical ≠ allowed by ToS. All three matter.

### How to Be a Mindful Scraper

Web scraping exists in a gray area — just because you *can* scrape something doesn't mean you *should*. Here are some best practices:

1. **Respect `robots.txt`**: Check it before you scrape. If a path is disallowed, don't go there.
2. **Rate-limit your requests**: Use tools like `backoff` and `time.sleep()` to space out your requests. A good rule of thumb: don't make requests faster than a human would browse.
3. **Scrape only what you need**: Don't download entire websites if you only need a few pages. Be surgical.
4. **Cache your results**: Save scraped data locally so you don't have to re-scrape the same pages over and over.
5. **Prefer APIs when available**: As we'll see shortly, APIs are almost always the better option — they're faster, more reliable, and the data provider *wants* you to use them.
6. **Don't scrape behind authentication** unless you have explicit permission — logging in programmatically to scrape is almost always a ToS violation.
7. **Think about the server on the other end**: Small websites with limited resources can be taken down by aggressive scraping. A university student's research project accidentally DDoS-ing a small nonprofit's website is a real thing that happens.

> **The golden rule of scraping**: Treat every website the way you'd want someone to treat *your* website.

### Fair Use and Scraped Data

- U.S. copyright law includes a **fair use** doctrine (17 U.S.C. § 107) that allows limited use of copyrighted material without permission. Courts evaluate four factors:
    1. **Purpose and character of the use** — Is it commercial or for nonprofit/educational purposes? *Transformative* uses (where you're creating something new, not just copying) are strongly favored.
    2. **Nature of the copyrighted work** — Factual content (statistics, prices, public records) gets less protection than creative content (articles, photos, fiction).
    3. **Amount used** — Scraping headlines and metadata is very different from scraping full article text. The less you take relative to the whole, the better.
    4. **Effect on the market** — Does your use substitute for the original? If people can use your scraped dataset instead of visiting the website, that weighs against fair use.

- **What this means for research scraping**:
    - Scraping *facts* (prices, dates, public statistics) is generally safe — facts aren't copyrightable
    - Scraping for *analysis and transformation* (sentiment analysis, building a model, creating summary statistics) is much stronger than scraping to republish
    - Scraping full copyrighted text (articles, books) and redistributing it is the riskiest use case
    - Academic and nonprofit research purposes weigh in your favor, but don't make you immune

- **The AI training debate**: This is the frontier of fair use law right now. Multiple lawsuits (e.g., *NYT v. OpenAI*, *Authors Guild v. OpenAI*) are testing whether scraping copyrighted text to train AI models counts as fair use. The outcomes of these cases will reshape the landscape significantly.

- **Practical advice for economists**: If you're scraping data for a research paper, you're generally on solid ground as long as you're (1) collecting facts rather than creative expression, (2) transforming the data through analysis, and (3) not redistributing the raw scraped content. When in doubt, consult your university's IRB or legal office.

Now let's make our loop that gets the content from the webpage:

In [160]:
base_url = "https://www.nytimes.com"

articles = []

for relative_link in url_list:
    site = requester(base_url + relative_link).content
    articles.append(site)
    print(f"Accessing {base_url + relative_link}")

Accessing https://www.nytimes.com/2026/02/18/us/politics/pirro-inquiry-video-democratic-lawmakers.html
Accessing https://www.nytimes.com/2026/02/17/us/politics/republicans-vote-fraud-id-midterms.html
Accessing https://www.nytimes.com/interactive/2026/02/18/us/politics/midterms-house-maps-redistricting.html
Accessing https://www.nytimes.com/2026/02/18/us/politics/democrats-trump-congress.html
Accessing https://www.nytimes.com/2026/02/18/us/politics/bhattacharya-kennedy-cdc-director.html
Accessing https://www.nytimes.com/2026/02/18/technology/meta-65-million-election-ai.html
Accessing https://www.nytimes.com/2026/02/18/us/politics/colorado-redistricting-house-map-2028.html
Accessing https://www.nytimes.com/2026/02/18/us/politics/pirro-inquiry-video-democratic-lawmakers.html
Accessing https://www.nytimes.com/2026/02/18/health/fda-moderna-flu-vaccine-mrna.html
Accessing https://www.nytimes.com/2026/02/18/business/energy-environment/uber-electric-vehicle-charging-stations.html
Accessing htt

Now let's sift through and see what we can find. 

In [162]:
BeautifulSoup(articles[0]).html

<html lang="en"><head><title>nytimes.com</title><style>#cmsg{animation: A 1.5s;}@keyframes A{0%{opacity:0;}99%{opacity:0;}100%{opacity:1;}}</style></head><body style="margin:0"><p id="cmsg">Please enable JS and disable any ad blocker</p><script data-cfasync="false">var dd={'rt':'c','cid':'AHrlqAAAAAMAal6qDhxbZHoAgjpovg==','hsh':'499AE34129FA4E4FABC31582C3075D','t':'bv','qp':'','s':17439,'e':'1c15d7fd75f57a4fe89ef40815689c8c09eb9efc2a1406e369ba764a915c420a6c9d32bf2fb3d986e1631cebd3969b82','host':'geo.captcha-delivery.com','cookie':'5Ut1qKvUmxBeHxEovAD3Y~4K_MD~uRemh7skRbEhSr_ImqMsYAqECgRXZtWVAmAWUR8AP0ohIpuF7xfryux1rUU6X36aBzonhPyda6Av~pb9PMGQHvkUQcAzSREYlWlY'}</script><script data-cfasync="false" src="https://ct.captcha-delivery.com/c.js"></script></body></html>

In [164]:
articles[0]

b'<html lang="en"><head><title>nytimes.com</title><style>#cmsg{animation: A 1.5s;}@keyframes A{0%{opacity:0;}99%{opacity:0;}100%{opacity:1;}}</style></head><body style="margin:0"><p id="cmsg">Please enable JS and disable any ad blocker</p><script data-cfasync="false">var dd={\'rt\':\'c\',\'cid\':\'AHrlqAAAAAMAal6qDhxbZHoAgjpovg==\',\'hsh\':\'499AE34129FA4E4FABC31582C3075D\',\'t\':\'bv\',\'qp\':\'\',\'s\':17439,\'e\':\'1c15d7fd75f57a4fe89ef40815689c8c09eb9efc2a1406e369ba764a915c420a6c9d32bf2fb3d986e1631cebd3969b82\',\'host\':\'geo.captcha-delivery.com\',\'cookie\':\'5Ut1qKvUmxBeHxEovAD3Y~4K_MD~uRemh7skRbEhSr_ImqMsYAqECgRXZtWVAmAWUR8AP0ohIpuF7xfryux1rUU6X36aBzonhPyda6Av~pb9PMGQHvkUQcAzSREYlWlY\'}</script><script data-cfasync="false" src="https://ct.captcha-delivery.com/c.js"></script></body></html>'

In [163]:
[BeautifulSoup(a).title for a in articles]

[<title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title data-rh="true">Tracking the Battle to Reshape Congress for the Midterms - The New York Times</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title>nytimes.com</title>,
 <title data-rh="true">Tracking the Battle to Reshape Congress for the Midterms - The New York Times</title>,
 <title>nytimes.com</title>]

- This is the issue with using libraries like `BeautifulSoup`, which are just there to parse through static html: as soon as there's anything *dynamic*, that requires javascript, things become tricky.
- If a website has a paywall or is accessing content dynamically through a server, we can no longer access the content
- But it does seem like we can at least extract article titles!
- Let's backtrack a bit and just scrape each section of the newspaper and extract each article title.

In [154]:
# get just the homepage to extract sections for the newspaper
url = "https://www.nytimes.com/"

r = requests.get(url)

soup = BeautifulSoup(r.text, "lxml")

# li = soup.find_all("li", class_="css-1bo3wsb")

# # let's only keep the hrefs that contain the parent "section"
# # For cleanliness don't consider absolute hrefs with nytimes.com included
# sections = []
# for l in li:
#     href = l.a.get("href")
#     if "section" in href and base_url not in href:
#         sections.append(href)

# sections

# Can we get the href from each of these?

# import io
# import pandas as pd
# pd.read_html(io.StringIO(section_table))

In [158]:
soup

<!DOCTYPE html>
<html class="nytapp-vi-homepage" data-nyt-compute-assignment="fallback" lang="en" xmlns:og="http://opengraphprotocol.org/schema/">
<head>
<meta charset="utf-8"/>
<title data-rh="true">The New York Times - Breaking News, US News, World News and Videos</title>
<meta content="Live news, investigations, opinion, photos and video by the journalists of The New York Times from more than 150 countries around the world. Subscribe for coverage of U.S. and international news, politics, business, technology, science, health, arts, sports and more." data-rh="true" name="description"/><meta content="https://www.nytimes.com" data-rh="true" property="og:url"/><meta content="website" data-rh="true" property="og:type"/><meta content="The New York Times - Breaking News, US News, World News and Videos" data-rh="true" property="og:title"/><meta content="Live news, investigations, opinion, photos and video by the journalists of The New York Times from more than 150 countries around the world

-  We were able to extract the sections of the NYTimes, now let's try to get the articles titles from each section

In [46]:
articles = []

for i, section_url in enumerate(sections):
    site = requester(base_url + section_url).content
    soup = BeautifulSoup(site)
    needed_divs = soup.find_all("article")
    for div in needed_divs:
        articles.append(div.a.text)


In [47]:
articles

[]

In [153]:
name_text = " ".join(articles)

stopwords = STOPWORDS.copy()

stopwords.update([
    "S",
    "U",
    "Back",
    "New",
    "York",
    "t",
    "Two",
    "Review",
    "One",
    "F",
    "Bill",
    "Alex",
    "Robert",
    "N",
    "Home",
    "Will",
])

wordcloud = WordCloud(
    background_color="white", collocations=False, stopwords=stopwords
).generate(name_text)

plt.imshow(wordcloud)

TypeError: sequence item 0: expected str instance, bytes found

## The Wonderful World of APIs

- Scraping websites should be usually seen as a last resort when there aren't any APIs available
- Websites don't tend to like scraping as it can increases traffic flow and slows down servers
- If a website or company is large enough, they may introduce an API (Application Programming Interface) to let you get information in a way that's a win-win
- Like say... the NY Times!
- Many organizations have APIs and python wrappers are available for those APIs.
- Using the Python packages for APIs are package-specific, but they all revolve around the same underlying structure: making URL queries to the web to extract information.

## Using REST APIs

- Technically, the term API is really a general term for anything that allows for software and eases programming
- Pandas is a library with an API, for instance
- In this case, though, an API is a REST (Representational State Transfer) API interface for making web requests and getting back information (usually in the form of JSON)
- You can break down web queries into three parts: GET, HEAD, PUT, POST, DELETE

![](images/1_R8Li_PHLFdB-VyMtl8G5_w.png)

- We'll really only need GET requests
- To use the NYTimes API, go to: https://developer.nytimes.com/
- Create an account
- Once you log-in go to the top-right, to the dropdown and click "Apps"
- Create a new app
- This will have an API key that you will need
- Under APIs, authorize `Books API`

In [42]:
api_key = "K9jZ2kcQHIucNredNfV7XocRoCj1s1B4"

r = requests.get(f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q=election&api-key={api_key}").content

articles = json.loads(r)

In [52]:
articles['response']['docs'][0]['abstract']

'The West Virginia Democrat could run for re-election to the Senate, make a third-party presidential bid or simply retire from politics. To his party’s consternation, he’s not ready to say which.'

In [53]:
# articles turned the json into a dictionary
# you can traverse the dictionary by using the `keys` method
articles.keys()
# extract the titles and lead paragraphs from each article

article_dict = {}

for article in articles['response']['docs']:
    article_dict.update({article['headline']['main'] : article['lead_paragraph']})


In [56]:
article_dict

{'Manchin Mulls His Political Future, Keeping Washington Guessing': 'Senator Joe Manchin III, the conservative West Virginia Democrat, was attending an event in his home state last month when he made a joke that quickly touched off the latest round of feverish speculation about his political future.',
 'G.O.P. Gets the Democratic Border Crisis It Wanted': 'When Gov. Greg Abbott of Texas began sending migrants and asylum seekers from the southwestern frontier to New York, Washington and Chicago, he vowed to bring the border to the Democratic cities he said were naïvely dismissing its costs.',
 'Ramaswamy’s Foreign Policy Approach Offers Rivals a Line of Attack': 'Republican presidential rivals, looking to blunt Vivek Ramaswamy’s rise in national primary polls ahead of the first primary debate on Wednesday, have seized on the political arena where the upstart entrepreneur has strayed far afield from his party’s thinkers: foreign policy.',
 'Ramaswamy Rides Wave of Support, but Rivals Are

In [54]:
def nytimes_search(query):
    api_key = "K9jZ2kcQHIucNredNfV7XocRoCj1s1B4"

    r = requests.get(f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&api-key={api_key}").content

    return json.loads(r)

In [55]:
nytimes_search('trump')

{'status': 'OK',
 'copyright': 'Copyright (c) 2023 The New York Times Company. All Rights Reserved.',
 'response': {'docs': [{'abstract': 'Donald J. Trump appeared in court as lawyers for New York’s attorney general, Letitia James, painted him as a fraudster. His lawyers said she was out to get the former president.',
    'web_url': 'https://www.nytimes.com/2023/10/02/nyregion/trump-fraud-trial-letitia-james.html',
    'snippet': 'Donald J. Trump appeared in court as lawyers for New York’s attorney general, Letitia James, painted him as a fraudster. His lawyers said she was out to get the former president.',
    'lead_paragraph': 'The trials of Donald J. Trump began Monday in a New York courtroom, where the former president arrived to fight the first of several government actions — a civil fraud case that imperils his company and threatens his image as a master of the business world.',
    'print_section': 'A',
    'print_page': '1',
    'source': 'The New York Times',
    'multimedia'

## Recap 

- So with the REST API we got what we were looking for!
- REST APIs are everywhere, you just need to find them and see if they are available. This makes extracting information from the web easy.
- The same ideas work for FRED, or twitter (although they might also have python packages, just search google: "python twitter api")
- But sometimes scraping is the only way
    - In that case, be careful, and be a good web-citizen: use backoffs and don't make too many requests, especially if the website is small.

## Web Scraping with Selenium

- Earlier, we saw that `BeautifulSoup` alone couldn't get past the NYTimes paywall — all we got back was "Please enable JS and disable any ad blocker"
- This is because `requests.get()` only downloads the **raw HTML** — it doesn't run any JavaScript
- Many modern websites load their content *dynamically* using JavaScript after the initial page load
- **Selenium** solves this by controlling an actual web browser (Chrome, Firefox, etc.) that executes JavaScript just like a real user would
- This means Selenium can handle:
    - JavaScript-rendered content
    - Pages that require clicking buttons or scrolling
    - Login forms and paywalls (with credentials)
    - Infinite scroll pages

### What is a Headless Browser?

- A **headless browser** is a web browser without a visible window — it runs in the background
- It does everything a normal browser does: loads pages, runs JavaScript, renders CSS — you just don't see it
- Think of it like a browser for robots: all the functionality, none of the GUI
- **Why headless?**
    - Much faster — no need to render pixels on screen
    - Uses less memory
    - Perfect for servers and automated scripts
    - Can still take screenshots if you need to debug
- Selenium can run in both **headed** (you see the browser) and **headless** (invisible) mode
- Headed mode is great for debugging — you can literally watch your script click through pages

### Setting Up Selenium

To install selenium:

`pip install selenium`

You also need a browser driver. Modern Selenium (v4+) handles this automatically with its built-in **Selenium Manager** — no need to manually download ChromeDriver anymore!

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
from bs4 import BeautifulSoup

### The Problem: JavaScript-Rendered Content

Let's use a website that's *designed* to demonstrate this problem: [quotes.toscrape.com/js/](https://quotes.toscrape.com/js/)

This site loads all its quotes using JavaScript. If you try to scrape it with `requests` + `BeautifulSoup`, you get... nothing.

In [9]:
# Try with requests + BeautifulSoup — this will FAIL to get quotes
url = "https://quotes.toscrape.com/js/"

r = requests.get(url)
soup = BeautifulSoup(r.text, "html.parser")

# Try to find the quotes
quotes = soup.find_all("div", class_="quote")
print(
    soup.find("script", src=None).text[:300]
    if soup.find("script", src=None)
    else "No inline script found"
)


    var data = [
    {
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ],
        "author": {
            "name": "Albert Einstein",
            "goodreads_link": "/author/show/9810.Albert_Einstein",
            "slug": "Alber


**Zero quotes found!** The HTML returned by `requests.get()` is just a skeleton with a JavaScript file that *would* generate the quotes — but `requests` doesn't run JavaScript. 

Compare this to the non-JS version of the same site:

In [10]:
# The non-JS version works fine with BeautifulSoup
url_static = "https://quotes.toscrape.com/"

r_static = requests.get(url_static)
soup_static = BeautifulSoup(r_static.text, "html.parser")

quotes_static = soup_static.find_all("div", class_="quote")
print(f"Quotes found on static version: {len(quotes_static)}")
for q in quotes_static[:3]:
    print(f"  - {q.find('span', class_='text').text[:80]}...")

Quotes found on static version: 10
  - “The world as we have created it is a process of our thinking. It cannot be chan...
  - “It is our choices, Harry, that show what we truly are, far more than our abilit...
  - “There are only two ways to live your life. One is as though nothing is a miracl...


### Selenium to the Rescue

Now let's use Selenium to scrape the **JavaScript version** of the same site. Selenium will launch a real browser, wait for JavaScript to execute, and *then* we grab the HTML.

In [11]:
# Set up Chrome in headless mode
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run without opening a visible window

# Create the browser instance
driver = webdriver.Chrome(options=chrome_options)

# Navigate to the JS-rendered page
driver.get("https://quotes.toscrape.com/js/")

# Wait for the quotes to load (up to 10 seconds)
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "quote"))
)

# Now grab the fully-rendered HTML and parse it with BeautifulSoup
soup_selenium = BeautifulSoup(driver.page_source, "html.parser")

quotes_selenium = soup_selenium.find_all("div", class_="quote")
print(f"Quotes found with Selenium: {len(quotes_selenium)}")
print()
for q in quotes_selenium[:5]:
    text = q.find("span", class_="text").text
    author = q.find("small", class_="author").text
    print(f'  "{text}" — {author}')

# Always close the browser when done!
driver.quit()

Quotes found with Selenium: 10

  "“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”" — Albert Einstein
  "“It is our choices, Harry, that show what we truly are, far more than our abilities.”" — J.K. Rowling
  "“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”" — Albert Einstein
  "“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”" — Jane Austen
  "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”" — Marilyn Monroe


### Selenium Can Also Interact with Pages

Selenium isn't just for reading — it can **click buttons**, **fill forms**, and **navigate** between pages. Let's scrape multiple pages by clicking the "Next" button.

In [12]:
# Scrape multiple pages by clicking "Next"
chrome_options = Options()
chrome_options.add_argument("--headless")
driver = webdriver.Chrome(options=chrome_options)

all_quotes = []

driver.get("https://quotes.toscrape.com/js/")

for page_num in range(1, 4):  # Scrape 3 pages
    # Wait for quotes to load
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, "quote"))
    )

    # Extract quotes from current page
    quote_elements = driver.find_elements(By.CLASS_NAME, "quote")
    for q in quote_elements:
        text = q.find_element(By.CLASS_NAME, "text").text
        author = q.find_element(By.CLASS_NAME, "author").text
        all_quotes.append({"text": text, "author": author})

    print(f"Page {page_num}: found {len(quote_elements)} quotes")

    # Click the "Next" button (if it exists)
    try:
        next_button = driver.find_element(By.CSS_SELECTOR, "li.next a")
        next_button.click()
    except:
        print("No more pages!")
        break

driver.quit()

print(f"\nTotal quotes scraped: {len(all_quotes)}")
print(f"Unique authors: {len(set(q['author'] for q in all_quotes))}")

Page 1: found 10 quotes
Page 2: found 10 quotes
Page 3: found 10 quotes

Total quotes scraped: 30
Unique authors: 20


### Putting It All Together: When to Use What

| Tool | Use When | Limitations |
|------|----------|-------------|
| `requests` + `BeautifulSoup` | Static HTML pages, simple APIs | No JavaScript, no interaction |
| `selenium` | JavaScript-rendered content, clicking/scrolling, login required | Slower, uses more memory |
| REST APIs | Data provider offers an API | Need API key, rate limits |

**General rule of thumb:**
1. First, check if there's an **API** — always prefer this
2. If no API, try `requests` + `BeautifulSoup` — fastest and simplest
3. If that fails (JavaScript content, paywalls), reach for `selenium`
4. Always be respectful: use delays between requests, check `robots.txt`, and don't hammer servers